In [84]:
import json, os, time, requests, ast
import pandas as pd
from pathlib import Path
from steam_web_api import Steam
from tqdm import tqdm
from dotenv import load_dotenv

#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', None)

In [85]:
load_dotenv()

True

In [ ]:
# https://github.com/bdefore/protondb-data

# this file updates monthly, adjust this as needed for fresh data

In [87]:
%%bash
curl -L -O -C - https://raw.githubusercontent.com/bdefore/protondb-data/master/reports/reports_apr1_2026.tar.gz

** Resuming transfer from byte position 64056754
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


In [88]:
%%bash
tar -xzvf reports_apr1_2026.tar.gz

reports_piiremoved.json


In [89]:
steam = Steam(os.environ.get("STEAM_API_KEY"))

player_json = steam.users.search_user(os.environ.get('STEAM_PROFILE'))

# https://store.steampowered.com/account/
# the steam ID is also listed under your account username at the top of the page
user_games_json = steam.users.get_owned_games(player_json['player']['steamid'])
user_games_df = pd.json_normalize(user_games_json['games'])

In [90]:
user_games_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   appid                        596 non-null    int64  
 1   name                         596 non-null    str    
 2   playtime_forever             596 non-null    int64  
 3   img_icon_url                 596 non-null    str    
 4   playtime_windows_forever     596 non-null    int64  
 5   playtime_mac_forever         596 non-null    int64  
 6   playtime_linux_forever       596 non-null    int64  
 7   playtime_deck_forever        596 non-null    int64  
 8   rtime_last_played            596 non-null    int64  
 9   content_descriptorids        153 non-null    object 
 10  playtime_disconnected        596 non-null    int64  
 11  has_community_visible_stats  474 non-null    object 
 12  has_leaderboards             70 non-null     object 
 13  playtime_2weeks              2 

In [91]:
user_games_df.head()

,appid,name,playtime_forever,img_icon_url,playtime_windows_forever,playtime_mac_forever,playtime_linux_forever,playtime_deck_forever,rtime_last_played,content_descriptorids,playtime_disconnected,has_community_visible_stats,has_leaderboards,playtime_2weeks
0,10,Counter-Strike,0,6b0312cda02f5f777efa2f3318c307ff9acafbb5,0,0,0,0,0,"[2, 5]",0,NaN,NaN,NaN
1,20,Team Fortress Classic,0,38ea7ebe3c1abbbbf4eabdbef174c41a972102b9,0,0,0,0,0,"[2, 5]",0,NaN,NaN,NaN
2,30,Day of Defeat,64,aadc0ce51ff6ba2042d633f8ec033b0de62091d0,0,0,0,0,1325318400,"[2, 5]",0,NaN,NaN,NaN
3,40,Deathmatch Classic,0,c525f76c8bc7353db4fd74b128c4ae2028426c2a,0,0,0,0,0,NaN,0,NaN,NaN,NaN
4,50,Half-Life: Opposing Force,0,04e81206c10e12416908c72c5f22aad411b3aeef,0,0,0,0,0,"[2, 5]",0,NaN,NaN,NaN


In [92]:
user_games_df[user_games_df['appid'] == 1076160]

,appid,name,playtime_forever,img_icon_url,playtime_windows_forever,playtime_mac_forever,playtime_linux_forever,playtime_deck_forever,rtime_last_played,content_descriptorids,playtime_disconnected,has_community_visible_stats,has_leaderboards,playtime_2weeks
553,1076160,Command: Modern Operations,5358,0f734b8c2b18433acc9eaeb8a0ea67139f369555,5358,0,0,0,1675035286,NaN,0,NaN,NaN,NaN


In [93]:
with open('reports_piiremoved.json', 'r') as f:
    jsonstr = json.load(f)

protondb_df = pd.json_normalize(jsonstr, sep="_")

In [94]:
protondb_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 356557 entries, 0 to 356556
Columns: 173 entries, timestamp to responses_followUp_inputFaults_bounding_0
dtypes: int64(1), object(59), str(113)
memory usage: 625.8+ MB


In [95]:
# cast as integer so we can do the join appropriately later
protondb_df['app_steam_appId'] = protondb_df['app_steam_appId'].astype(int)

In [96]:
print(protondb_df['app_steam_appId'].dtype)

int64


In [97]:
protondb_df.head()

,timestamp,app_steam_appId,app_title,responses_answerToWhatGame,responses_appSelectionMethod,responses_installs,responses_notes_extra,responses_notes_verdict,responses_protonVersion,responses_type,...,responses_followUp_controlLayoutCustomization_enableGripButtons_0,responses_followUp_windowingFaults_switching_0,responses_launchFlagsUsed_noXim,responses_followUp_audioFaults_outOfSync_0,responses_customizationsUsed_lutris_0,responses_followUp_audioFaults_borked_0,systemInfo_steamRuntimeVersion,systemInfo_xWindowManager,responses_frameRate,responses_followUp_inputFaults_bounding_0
0,1572299227,352620,Porcunipine,352620,libraryLookup,no,oh noes,is le borked,Default,steamPlay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1572304395,219990,Grim Dawn,219990,NaN,yes,NaN,out the box,Default,steamPlay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1572305157,367500,Dragon's Dogma: Dark Arisen,367500,libraryLookup,yes,Whilst it is possible to use pawns made by oth...,"With D9VK enabled, the game runs almost perfec...",Default,steamPlay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1572305804,251570,7 Days to Die,251570,NaN,yes,start option on unity launcher \n-force-d3d11 ...,with preset on high i have 55-60 fps,Default,steamPlay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1572306754,292030,The Witcher 3: Wild Hunt,292030,libraryLookup,yes,NaN,Runs perfectly,Default,steamPlay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
cached_steamwishlist_data = Path("cached_steamwishlist_"+os.environ.get('STEAM_PROFILE')+".json")

jsonstr = []

if cached_steamwishlist_data.is_file():
    with open(cached_steamwishlist_data, 'r') as f:
        jsonstr = json.load(f)
    
    # use sets as they are O(1) lookups
    already_cached_appids = set([item['appid'] for item in jsonstr])
else:
    already_cached_appids = set()
    
print(f"Number of wishlist cached appids: {len(already_cached_appids)}")

Number of wishlist cached appids: 291


In [ ]:
# loop and download Steam wishlist data
wishlist_list = []
# https://steamapi.xpaw.me/IWishlistService
response = requests.get("https://api.steampowered.com/IWishlistService/GetWishlist/v1/?key="+os.environ.get('STEAM_API_KEY')+"&steamid="+player_json['player']['steamid'])

# this is also an option if we don't want to do it manually with the requests library
# user_wishlist = steam.users.get_profile_wishlist(player_json['player']['steamid'])

if response.status_code == 200:
    json_resp = response.json()
    if json_resp:
        # add the appid into the responses so we can reference them properly
        wishlist_list = list(json_resp['response']['items'])
    
# note that the number displayed in the UI sometimes can be larger than the number here
# this is because some games you previously wishlisted become unlisted, this script can help
# cleanup old games that are no longer on Steam
# https://steamcommunity.com/sharedfiles/filedetails/?id=1746978201
print(f"Number of wishlisted games: {len(wishlist_list)}")

# lookup game names using appids
for game in tqdm(wishlist_list):
    if game['appid'] not in already_cached_appids:
        gamedetails = steam.apps.get_app_details(game['appid'], country="US", filters="basic,metacritic")
        for app_id, details in gamedetails.items():
            if details.get('success'):
                game['name'] = details['data']['name']
                meta = details['data'].get('metacritic')
                if meta:
                    game['metacritic'] = meta.get('score')
                else:
                    game['metacritic'] = None
                jsonstr.append(game)
        time.sleep(2) # sleep for a few seconds so we don't slam the Steam API

# remove any wishlist items that have been cached but no longer are in the wishlist
for game in tqdm(jsonstr):
    if not any(game['appid'] in d.values() for d in wishlist_list):
        print(f"{game} no longer in wishlist")
        jsonstr.remove(game)

# cache the file so we don't need to download it again in the future
with open(cached_steamwishlist_data, 'w') as json_file:
    json.dump(jsonstr, json_file, indent=4)  # indent=4 for pretty printing

Number of wishlisted games: 291


100%|██████████| 291/291 [00:00<00:00, 26702.45it/s]


In [100]:
with open(cached_steamwishlist_data, 'r') as f:
    jsonstr = json.load(f)

wishlist_df = pd.json_normalize(jsonstr)

In [101]:
# cast as integer so we can do the join appropriately later
wishlist_df['appid'] = wishlist_df['appid'].astype(int)
# convert unix epoch time to timestamp
wishlist_df['date_added'] = pd.to_datetime(wishlist_df['date_added'], unit='s')

In [102]:
wishlist_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 291 entries, 0 to 290
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype        
---  ------      --------------  -----        
 0   appid       291 non-null    int64        
 1   priority    291 non-null    int64        
 2   date_added  291 non-null    datetime64[s]
 3   name        291 non-null    str          
 4   metacritic  91 non-null     float64      
dtypes: datetime64[s](1), float64(1), int64(2), str(1)
memory usage: 16.0 KB


In [103]:
wishlist_df.sort_values(by='date_added', ascending=False).head()

,appid,priority,date_added,name,metacritic
290,4585960,0,2026-04-10 18:47:11,Eavor,NaN
289,1963610,0,2026-04-09 16:31:10,Road to Vostok,NaN
186,1488490,0,2026-03-29 16:57:41,Nivalis,NaN
221,1846700,0,2026-03-29 16:56:51,Witchbrook,NaN
227,1944880,0,2026-03-29 16:15:33,Drop Command,NaN


In [104]:
wishlist_df.head()

,appid,priority,date_added,name,metacritic
0,210970,18,2016-01-25 22:46:17,The Witness,87.0
1,16900,4,2018-09-08 01:07:48,GROUND BRANCH,NaN
2,201810,46,2014-11-07 00:48:49,Wolfenstein: The New Order,81.0
3,205730,82,2012-07-21 01:51:20,Insanely Twisted Shadow Planet,76.0
4,211400,83,2012-10-27 17:14:32,Deadlight,78.0


In [105]:
print(list(protondb_df.columns))

['timestamp', 'app_steam_appId', 'app_title', 'responses_answerToWhatGame', 'responses_appSelectionMethod', 'responses_installs', 'responses_notes_extra', 'responses_notes_verdict', 'responses_protonVersion', 'responses_type', 'responses_verdict', 'systemInfo_cpu', 'systemInfo_gpu', 'systemInfo_gpuDriver', 'systemInfo_kernel', 'systemInfo_os', 'systemInfo_ram', 'responses_audioFaults', 'responses_duration', 'responses_extra', 'responses_graphicalFaults', 'responses_inputFaults', 'responses_isImpactedByAntiCheat', 'responses_isMultiplayerImportant', 'responses_localMultiplayerAttempted', 'responses_onlineMultiplayerAttempted', 'responses_opens', 'responses_performanceFaults', 'responses_saveGameFaults', 'responses_significantBugs', 'responses_stabilityFaults', 'responses_startsPlay', 'responses_windowingFaults', 'responses_notes_significantBugs', 'responses_followUp_isImpactedByAntiCheat_easyAntiCheat', 'responses_followUp_audioFaults_crackling', 'responses_notes_audioFaults', 'response

In [106]:
protondb_df['responses_verdict'].unique()

<ArrowStringArray>
['no', 'yes']
Length: 2, dtype: str

In [107]:
grouped_protondb_df = protondb_df.groupby(['app_steam_appId', 'responses_verdict']).size().reset_index(name='count')

In [108]:
grouped_protondb_df.head()

,app_steam_appId,responses_verdict,count
0,10,no,18
1,10,yes,57
2,20,no,24
3,20,yes,13
4,30,no,1


In [109]:
# Group by app_steam_appId and calculate the total count and yes count
total_counts = grouped_protondb_df.groupby('app_steam_appId')['count'].sum().reset_index(name='total_protondb_submissions')
yes_counts = grouped_protondb_df[grouped_protondb_df['responses_verdict'] == 'yes'].groupby('app_steam_appId')['count'].sum().reset_index(name='supported_verdict_count')

# Merge the total and yes counts
grouped_protondb_percentages_df = pd.merge(total_counts, yes_counts, on='app_steam_appId', how='left')

# Calculate the percentage of yes responses
grouped_protondb_percentages_df['protondb_supported_percentage'] = round((grouped_protondb_percentages_df['supported_verdict_count'] / grouped_protondb_percentages_df['total_protondb_submissions']) * 100, 2)

grouped_protondb_percentages_df['protondb_supported_percentage'] = grouped_protondb_percentages_df['protondb_supported_percentage'].fillna(0)

# convert to integer
grouped_protondb_percentages_df['supported_verdict_count'] = grouped_protondb_percentages_df['supported_verdict_count'].fillna(0)
grouped_protondb_percentages_df['supported_verdict_count'] = grouped_protondb_percentages_df['supported_verdict_count'].astype(int)

In [110]:
grouped_protondb_percentages_df.head()

,app_steam_appId,total_protondb_submissions,supported_verdict_count,protondb_supported_percentage
0,10,75,57,76.00
1,20,37,13,35.14
2,30,12,11,91.67
3,40,9,8,88.89
4,50,26,26,100.00


In [111]:
cached_deckverified_data = Path("cached_deckverified_"+os.environ.get('STEAM_PROFILE')+".json")

In [112]:
if cached_deckverified_data.is_file():
    with open(cached_deckverified_data, 'r') as f:
        jsonstr = json.load(f)
    
    # use sets as they are O(1) lookups
    already_cached_appids = set([item['appid'] for item in jsonstr])
else:
    already_cached_appids = set()
    
print(f"Number of cached appids: {len(already_cached_appids)}")

Number of cached appids: 887


In [113]:
# this can take a while to run depending on how many games you have
# loop and download steam deck verified data
# e.g. https://www.protondb.com/proxy/steam/deck-verified?nAppID=550
verified_list = []
for appid in tqdm(list(user_games_df['appid']) + list(wishlist_df['appid'])):
    if appid not in already_cached_appids:
        response = requests.get("https://www.protondb.com/proxy/steam/deck-verified?nAppID="+str(appid))

        if response.status_code == 200:
            json_resp = response.json()
            if (json_resp['success'] == 1 and 'resolved_category' in json_resp['results']):
                # clean up and remove null entries
                insert_str = str(json_resp['results']).replace(", 'search_id': None", "")
                insert_dict = ast.literal_eval(insert_str)
                # look up metacritic information if it's an owned game
                if appid not in wishlist_df['appid']:
                    gamedetails = steam.apps.get_app_details(appid, country="US", filters="basic,metacritic")
                    for app_id, details in gamedetails.items():
                        if details.get('success'):
                            meta = details['data'].get('metacritic')
                            if meta:
                                insert_dict['metacritic'] = meta.get('score')
                            else:
                                insert_dict['metacritic'] = None
                else:
                    # lookup metacritic score from wishlist
                    match = wishlist_df.loc[wishlist_df["appid"] == appid, "metacritic"]
                    if not match.empty:
                        insert_dict["metacritic"] = match.iloc[0]
                    else:
                        insert_dict["metacritic"] = None
                verified_list.append(insert_dict)
            elif json_resp['success'] == 1:
                # we got a successful response but the entry isn't a game or entry anymore...
                # save a blank entry so we can skip it later
                inject_dict = {
                    "appid": appid,
                    "resolved_category": 0,
                    "resolved_items": [],
                    "steam_deck_blog_url": "",
                    "metacritic": None
                }
                verified_list.append(inject_dict)
            else:
                print(f"Warning skipped appid: {appid}")
                print(json_resp)            
                    
        time.sleep(2) # sleep for a few seconds so we don't slam the protondb servers    

100%|██████████| 887/887 [00:00<00:00, 4697408.65it/s]


In [114]:
# cache the file so we don't need to download it in the future
# note that sometimes the official steam deck verified status can change, however,
# this probably happens rarely, if needed, every so often you can delete this
# cache and re-download to confirm you have up-to-date information on steam deck status
if cached_deckverified_data.is_file():
    with open(cached_deckverified_data, 'r') as f:
        jsonstr = json.load(f)
    verified_list = verified_list + jsonstr

with open(cached_deckverified_data, 'w') as json_file:
    json.dump(verified_list, json_file, indent=4)  # indent=4 for pretty printing

In [115]:
len(verified_list)

887

In [116]:
with open(cached_deckverified_data, 'r') as f:
    jsonstr = json.load(f)

flattened_data = []
for entry in jsonstr:
    try:
        if entry['resolved_items']:
            # Flatten resolved_items if they exist
            flattened_entry = pd.json_normalize(
                entry,
                record_path='resolved_items',  # Unnest resolved_items
                meta=['appid', 'resolved_category', 'steam_deck_blog_url', 'metacritic'],  # Keep the parent fields
                errors='raise'  # Raise errors so we can catch them
            )
        else:
            # If resolved_items is empty, create a single row manually with None for the item fields
            flattened_entry = pd.DataFrame([{
                'display_type': None,
                'loc_token': None,
                'appid': entry['appid'],
                'resolved_category': entry['resolved_category'],
                'steam_deck_blog_url': entry['steam_deck_blog_url'],
                'metacritic': entry['metacritic']
            }])
            
        flattened_data.append(flattened_entry)
    except Exception as e:
        # Print the problematic entry and the error message
        print(f"Error processing entry with appid: {entry.get('appid')}")
        print(f"Error message: {e}")

deck_verified_df = pd.concat(flattened_data, ignore_index=True)

In [117]:
deck_verified_df['appid'].nunique()

887

In [118]:
# cast as integer so we can do the join appropriately later
deck_verified_df['appid'] = deck_verified_df['appid'].astype(int)

In [119]:
deck_verified_df.drop(columns=['loc_token', 'display_type', 'steam_deck_blog_url'], inplace=True)
deck_verified_df.drop_duplicates(inplace=True)

In [120]:
deck_verified_df.head()

,appid,resolved_category,metacritic
0,10,2,None
5,20,2,None
11,30,2,None
16,40,2,None
21,50,2,None


In [121]:
deck_verified_df["metacritic"] = pd.to_numeric(deck_verified_df["metacritic"], errors="coerce")

deck_verified_df['resolved_category'] = deck_verified_df['resolved_category'].astype(str)
# 0 = No Steam Deck Information
# 1 = Steam Deck Unsupported
# 2 = Steam Deck Playable
# 3 = Steam Deck Verified
deck_verified_df['resolved_category'] = deck_verified_df['resolved_category'].map({"0": "No Steam Deck Information",
                                                                                               "1": "Steam Deck Unsupported",
                                                                                               "2": "Steam Deck Playable",
                                                                                               "3": "Steam Deck Verified"})

In [122]:
deck_verified_df.info()

<class 'pandas.DataFrame'>
Index: 887 entries, 0 to 3408
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   appid              887 non-null    int64  
 1   resolved_category  887 non-null    str    
 2   metacritic         487 non-null    float64
dtypes: float64(1), int64(1), str(1)
memory usage: 45.3 KB


In [123]:
deck_verified_df.groupby(['resolved_category']).size().reset_index()

,resolved_category,0
0,No Steam Deck Information,131
1,Steam Deck Playable,431
2,Steam Deck Unsupported,101
3,Steam Deck Verified,224


In [124]:
deck_verified_df.head()

,appid,resolved_category,metacritic
0,10,Steam Deck Playable,NaN
5,20,Steam Deck Playable,NaN
11,30,Steam Deck Playable,NaN
16,40,Steam Deck Playable,NaN
21,50,Steam Deck Playable,NaN


In [125]:
user_games_verified_df = pd.merge(user_games_df, deck_verified_df, how='left', left_on='appid', right_on='appid')

In [126]:
user_games_verified_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   appid                        596 non-null    int64  
 1   name                         596 non-null    str    
 2   playtime_forever             596 non-null    int64  
 3   img_icon_url                 596 non-null    str    
 4   playtime_windows_forever     596 non-null    int64  
 5   playtime_mac_forever         596 non-null    int64  
 6   playtime_linux_forever       596 non-null    int64  
 7   playtime_deck_forever        596 non-null    int64  
 8   rtime_last_played            596 non-null    int64  
 9   content_descriptorids        153 non-null    object 
 10  playtime_disconnected        596 non-null    int64  
 11  has_community_visible_stats  474 non-null    object 
 12  has_leaderboards             70 non-null     object 
 13  playtime_2weeks              2 

In [128]:
user_games_verified_df.groupby(['resolved_category']).size().reset_index()

,resolved_category,0
0,No Steam Deck Information,82
1,Steam Deck Playable,290
2,Steam Deck Unsupported,77
3,Steam Deck Verified,147


In [129]:
user_games_verified_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   appid                        596 non-null    int64  
 1   name                         596 non-null    str    
 2   playtime_forever             596 non-null    int64  
 3   img_icon_url                 596 non-null    str    
 4   playtime_windows_forever     596 non-null    int64  
 5   playtime_mac_forever         596 non-null    int64  
 6   playtime_linux_forever       596 non-null    int64  
 7   playtime_deck_forever        596 non-null    int64  
 8   rtime_last_played            596 non-null    int64  
 9   content_descriptorids        153 non-null    object 
 10  playtime_disconnected        596 non-null    int64  
 11  has_community_visible_stats  474 non-null    object 
 12  has_leaderboards             70 non-null     object 
 13  playtime_2weeks              2 

In [130]:
grouped_df = user_games_verified_df[['appid', 'name', 'resolved_category', 'playtime_forever', 'metacritic']]

In [131]:
grouped_df = pd.merge(grouped_df, grouped_protondb_percentages_df, how='left', left_on='appid', right_on='app_steam_appId')
grouped_df['resolved_category'] = grouped_df['resolved_category'].astype(str)
grouped_df.drop(columns='app_steam_appId', inplace=True)

In [133]:
grouped_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 596 entries, 0 to 595
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   appid                          596 non-null    int64  
 1   name                           596 non-null    str    
 2   resolved_category              596 non-null    str    
 3   playtime_forever               596 non-null    int64  
 4   metacritic                     396 non-null    float64
 5   total_protondb_submissions     580 non-null    float64
 6   supported_verdict_count        580 non-null    float64
 7   protondb_supported_percentage  580 non-null    float64
dtypes: float64(4), int64(2), str(2)
memory usage: 60.0 KB


In [134]:
grouped_df.head(n=10)

,appid,name,resolved_category,playtime_forever,metacritic,total_protondb_submissions,supported_verdict_count,protondb_supported_percentage
0,10,Counter-Strike,Steam Deck Playable,0,NaN,75.0,57.0,76.00
1,20,Team Fortress Classic,Steam Deck Playable,0,NaN,37.0,13.0,35.14
2,30,Day of Defeat,Steam Deck Playable,64,NaN,12.0,11.0,91.67
3,40,Deathmatch Classic,Steam Deck Playable,0,NaN,9.0,8.0,88.89
4,50,Half-Life: Opposing Force,Steam Deck Playable,0,NaN,26.0,26.0,100.00
5,60,Ricochet,Steam Deck Playable,0,NaN,13.0,12.0,92.31
6,70,Half-Life,Steam Deck Verified,0,NaN,156.0,137.0,87.82
7,130,Half-Life: Blue Shift,Steam Deck Playable,0,NaN,23.0,22.0,95.65
8,220,Half-Life 2,Steam Deck Verified,795,NaN,191.0,181.0,94.76
9,240,Counter-Strike: Source,Steam Deck Playable,71,NaN,138.0,115.0,83.33


In [135]:
grouped_df.sort_values(by='playtime_forever', ascending=False).head(n=20)

,appid,name,resolved_category,playtime_forever,metacritic,total_protondb_submissions,supported_verdict_count,protondb_supported_percentage
459,427520,Factorio,Steam Deck Verified,65533,90.0,304.0,295.0,97.04
241,218620,PAYDAY 2,Steam Deck Verified,38293,79.0,316.0,272.0,86.08
361,286160,Tabletop Simulator,Steam Deck Unsupported,22517,NaN,102.0,91.0,89.22
172,107410,Arma 3,Steam Deck Unsupported,22393,74.0,284.0,242.0,85.21
492,526870,Satisfactory,Steam Deck Verified,18718,91.0,1201.0,1042.0,86.76
415,359320,Elite Dangerous,Steam Deck Playable,14354,80.0,773.0,606.0,78.40
450,413150,Stardew Valley,Steam Deck Verified,13600,89.0,343.0,338.0,98.54
271,232090,Killing Floor 2,Steam Deck Playable,12869,75.0,273.0,244.0,89.38
311,251570,7 Days to Die,Steam Deck Verified,10634,NaN,197.0,159.0,80.71
314,252490,Rust,Steam Deck Unsupported,9556,69.0,277.0,104.0,37.55


In [136]:
grouped_df.groupby(['resolved_category']).size().reset_index()

,resolved_category,0
0,No Steam Deck Information,82
1,Steam Deck Playable,290
2,Steam Deck Unsupported,77
3,Steam Deck Verified,147


In [137]:
grouped_df[(grouped_df['resolved_category'] == "No Steam Deck Information") | (grouped_df['resolved_category'] == "Steam Deck Unsupported")].sort_values(by='playtime_forever', ascending=False).head(n=len(grouped_df))

,appid,name,resolved_category,playtime_forever,metacritic,total_protondb_submissions,supported_verdict_count,protondb_supported_percentage
361,286160,Tabletop Simulator,Steam Deck Unsupported,22517,NaN,102.0,91.0,89.22
172,107410,Arma 3,Steam Deck Unsupported,22393,74.0,284.0,242.0,85.21
314,252490,Rust,Steam Deck Unsupported,9556,69.0,277.0,104.0,37.55
553,1076160,Command: Modern Operations,Steam Deck Unsupported,5358,NaN,25.0,8.0,32.00
395,321410,Command: Modern Air / Naval Operations WOTY,No Steam Deck Information,4051,NaN,3.0,0.0,0.00
...,...,...,...,...,...,...,...,...
523,648340,Harald,No Steam Deck Information,0,NaN,NaN,NaN,NaN
517,624690,NEXT JUMP: Shmup Tactics,No Steam Deck Information,0,NaN,1.0,1.0,100.00
539,813540,Scheming Through The Zombie Apocalypse: The Be...,No Steam Deck Information,0,NaN,4.0,4.0,100.00
587,2362420,Mass Effect 2 (2010) Edition,Steam Deck Unsupported,0,94.0,9.0,5.0,55.56


In [138]:
wishlist_verified_df = pd.merge(wishlist_df, deck_verified_df, how='left', left_on='appid', right_on='appid')

wishlist_verified_df["metacritic"] = wishlist_verified_df["metacritic_x"].combine_first(wishlist_verified_df["metacritic_y"])

wishlist_verified_df = wishlist_verified_df.drop(columns=["metacritic_x", "metacritic_y"])

In [139]:
wishlist_verified_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 291 entries, 0 to 290
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   appid              291 non-null    int64        
 1   priority           291 non-null    int64        
 2   date_added         291 non-null    datetime64[s]
 3   name               291 non-null    str          
 4   resolved_category  291 non-null    str          
 5   metacritic         91 non-null     float64      
dtypes: datetime64[s](1), float64(1), int64(2), str(2)
memory usage: 24.0 KB


In [140]:
wishlist_verified_df.head()

,appid,priority,date_added,name,resolved_category,metacritic
0,210970,18,2016-01-25 22:46:17,The Witness,Steam Deck Playable,87.0
1,16900,4,2018-09-08 01:07:48,GROUND BRANCH,Steam Deck Playable,NaN
2,201810,46,2014-11-07 00:48:49,Wolfenstein: The New Order,Steam Deck Verified,81.0
3,205730,82,2012-07-21 01:51:20,Insanely Twisted Shadow Planet,Steam Deck Unsupported,76.0
4,211400,83,2012-10-27 17:14:32,Deadlight,Steam Deck Playable,78.0


In [141]:
wishlist_verified_df.groupby(['resolved_category']).size().reset_index()

,resolved_category,0
0,No Steam Deck Information,49
1,Steam Deck Playable,141
2,Steam Deck Unsupported,24
3,Steam Deck Verified,77


In [142]:
wishlist_verified_grouped_df = wishlist_verified_df[['appid', 'name', 'resolved_category', 'date_added', 'metacritic']]

In [143]:
wishlist_verified_grouped_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 291 entries, 0 to 290
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype        
---  ------             --------------  -----        
 0   appid              291 non-null    int64        
 1   name               291 non-null    str          
 2   resolved_category  291 non-null    str          
 3   date_added         291 non-null    datetime64[s]
 4   metacritic         91 non-null     float64      
dtypes: datetime64[s](1), float64(1), int64(1), str(2)
memory usage: 21.7 KB


In [144]:
wishlist_verified_grouped_df.head()

,appid,name,resolved_category,date_added,metacritic
0,210970,The Witness,Steam Deck Playable,2016-01-25 22:46:17,87.0
1,16900,GROUND BRANCH,Steam Deck Playable,2018-09-08 01:07:48,NaN
2,201810,Wolfenstein: The New Order,Steam Deck Verified,2014-11-07 00:48:49,81.0
3,205730,Insanely Twisted Shadow Planet,Steam Deck Unsupported,2012-07-21 01:51:20,76.0
4,211400,Deadlight,Steam Deck Playable,2012-10-27 17:14:32,78.0


In [145]:
wishlist_verified_grouped_df.groupby(['resolved_category']).size().reset_index()

,resolved_category,0
0,No Steam Deck Information,49
1,Steam Deck Playable,141
2,Steam Deck Unsupported,24
3,Steam Deck Verified,77


In [146]:
wishlist_verified_grouped_df = pd.merge(wishlist_verified_grouped_df, grouped_protondb_percentages_df, how='left', left_on='appid', right_on='app_steam_appId')
wishlist_verified_grouped_df.drop(columns='app_steam_appId', inplace=True)

In [147]:
wishlist_verified_grouped_df.sort_values(by='date_added', ascending=False).head()

,appid,name,resolved_category,date_added,metacritic,total_protondb_submissions,supported_verdict_count,protondb_supported_percentage
290,4585960,Eavor,No Steam Deck Information,2026-04-10 18:47:11,NaN,NaN,NaN,NaN
289,1963610,Road to Vostok,Steam Deck Playable,2026-04-09 16:31:10,NaN,20.0,18.0,90.0
186,1488490,Nivalis,No Steam Deck Information,2026-03-29 16:57:41,NaN,NaN,NaN,NaN
221,1846700,Witchbrook,No Steam Deck Information,2026-03-29 16:56:51,NaN,NaN,NaN,NaN
227,1944880,Drop Command,No Steam Deck Information,2026-03-29 16:15:33,NaN,NaN,NaN,NaN


In [148]:
grouped_df['on_wishlist'] = "No"
wishlist_verified_grouped_df['on_wishlist'] = "Yes"

df_union = pd.concat([grouped_df, wishlist_verified_grouped_df], ignore_index=True)

# fill columns where there are NaN values with zeros
df_union['total_protondb_submissions'] = df_union['total_protondb_submissions'].fillna(0)
df_union['supported_verdict_count'] = df_union['supported_verdict_count'].fillna(0)
df_union['protondb_supported_percentage'] = df_union['protondb_supported_percentage'].fillna(0)
df_union['playtime_forever'] = df_union['playtime_forever'].fillna(0)
df_union['metacritic'] = df_union['metacritic'].fillna(0)

df_union.to_parquet("data.parquet")

In [150]:
df_union.info()

<class 'pandas.DataFrame'>
RangeIndex: 887 entries, 0 to 886
Data columns (total 10 columns):
 #   Column                         Non-Null Count  Dtype        
---  ------                         --------------  -----        
 0   appid                          887 non-null    int64        
 1   name                           887 non-null    str          
 2   resolved_category              887 non-null    str          
 3   playtime_forever               887 non-null    float64      
 4   metacritic                     887 non-null    float64      
 5   total_protondb_submissions     887 non-null    float64      
 6   supported_verdict_count        887 non-null    float64      
 7   protondb_supported_percentage  887 non-null    float64      
 8   on_wishlist                    887 non-null    str          
 9   date_added                     291 non-null    datetime64[s]
dtypes: datetime64[s](1), float64(5), int64(1), str(3)
memory usage: 104.4 KB


In [151]:
df_union.head()

,appid,name,resolved_category,playtime_forever,metacritic,total_protondb_submissions,supported_verdict_count,protondb_supported_percentage,on_wishlist,date_added
0,10,Counter-Strike,Steam Deck Playable,0.0,0.0,75.0,57.0,76.00,No,NaT
1,20,Team Fortress Classic,Steam Deck Playable,0.0,0.0,37.0,13.0,35.14,No,NaT
2,30,Day of Defeat,Steam Deck Playable,64.0,0.0,12.0,11.0,91.67,No,NaT
3,40,Deathmatch Classic,Steam Deck Playable,0.0,0.0,9.0,8.0,88.89,No,NaT
4,50,Half-Life: Opposing Force,Steam Deck Playable,0.0,0.0,26.0,26.0,100.00,No,NaT
